# Analisis Prediktif Kemenangan Tim — LCK 2025 (versi revisi)

**Perubahan dari versi sebelumnya (menjawab masukan reviewer soal temporal validity):**

1. **Model utama hanya memakai fitur numerik bertimestamp** (selisih gold, experience, CS, kill, assist).
   Seluruh indikator objektif biner (first blood, first dragon, void grub, first herald) dikeluarkan dari model utama.
2. **Indikator biner dipindah ke analisis sensitivitas** (`objektif=True`), untuk menguji apakah kesimpulan bergantung pada pengeluarannya.
3. **Sel baru: audit first baron / Atakhan** pada snapshot menit 25 (Langkah 7), dengan varian terpisah.
4. **Sel baru: uji proporsi dan Fisher** untuk subkelompok playoff, dan penampil hyperparameter terpilih (bahan Tabel III).
5. Gambar versi pertama (yang masih satu bahasa) dihapus karena sudah digantikan versi dwibahasa.

Semua angka di naskah harus diambil ulang dari hasil notebook ini.

In [91]:
import warnings

import numpy as np
import pandas as pd

# sklearn memunculkan UserWarning tentang `delayed`/`Parallel` ketika ada
# paralelisasi bersarang. Sifatnya kosmetik dan tidak memengaruhi hasil.
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

CSV_MENTAH = "oracles_elixir_2025.csv"
LIGA = "LCK"
SISI = "Blue"
TARGET = "result"
SNAPSHOTS = [10, 15, 20, 25]

SPLIT_LATIH = ["Cup", "Rounds 1-2"]
SPLIT_UJI = ["Rounds 3-5"]

RANDOM_STATE = 42

---
## Langkah 1 — Praproses data

Filter liga LCK dan baris tim, verifikasi kelengkapan, lalu ambil satu baris
per pertandingan untuk menghindari *pseudo-replication*.

In [92]:
kolom = [
    "gameid", "league", "year", "split", "playoffs", "date", "patch",
    "position", "side", "teamname", "result", "gamelength", "datacompleteness",
    "firstblood", "firstdragon", "firstherald", "firsttower", "firstbaron",
    "void_grubs", "opp_void_grubs", "atakhans", "opp_atakhans",
    "towers", "opp_towers", "dragons", "opp_dragons", "barons", "opp_barons",
    "dpm", "totalgold", "visionscore",
]
for t in SNAPSHOTS:
    kolom += [
        f"golddiffat{t}", f"xpdiffat{t}", f"csdiffat{t}",
        f"killsat{t}", f"opp_killsat{t}",
        f"assistsat{t}", f"opp_assistsat{t}",
    ]

tersedia = pd.read_csv(CSV_MENTAH, nrows=0).columns
df = pd.read_csv(CSV_MENTAH, usecols=[k for k in kolom if k in tersedia], low_memory=False)

df_tim = df[(df["league"] == LIGA) & (df["position"] == "team")].copy()
df_tim["date"] = pd.to_datetime(df_tim["date"])

print(f"baris tim          : {len(df_tim)}")
print(f"pertandingan unik  : {df_tim['gameid'].nunique()}")
print(f"baris tidak lengkap: {(df_tim['datacompleteness'] != 'complete').sum()}")
print(f"durasi terpendek   : {df_tim['gamelength'].min()/60:.1f} menit")
print(f"versi patch        : {df_tim['patch'].nunique()} ({df_tim['patch'].min()}-{df_tim['patch'].max()})")

baris tim          : 1110
pertandingan unik  : 555
baris tidak lengkap: 0
durasi terpendek   : 21.9 menit
versi patch        : 13 (15.01-15.17)


In [93]:
print("pertandingan per fase:")
print(df_tim.groupby("split")["gameid"].nunique().to_string())

print("\npertandingan playoff per fase:")
print(df_tim[df_tim["playoffs"] == 1].groupby("split")["gameid"].nunique().to_string())

pertandingan per fase:
split
Cup           109
Rounds 1-2    240
Rounds 3-5    206

pertandingan playoff per fase:
split
Rounds 1-2    18
Rounds 3-5    61


In [94]:
df_tim = df_tim[df_tim["datacompleteness"] == "complete"]

base = (df_tim[df_tim["side"] == SISI]
        .drop_duplicates(subset="gameid", keep="first")
        .sort_values("date")
        .reset_index(drop=True))

for t in SNAPSHOTS:
    base[f"killdiffat{t}"] = base[f"killsat{t}"] - base[f"opp_killsat{t}"]
    base[f"assistdiffat{t}"] = base[f"assistsat{t}"] - base[f"opp_assistsat{t}"]

base["grubdiff"] = base["void_grubs"] - base["opp_void_grubs"]
base["atakhandiff"] = base["atakhans"] - base["opp_atakhans"]
base["towerdiff"] = base["towers"] - base["opp_towers"]
base["dragondiff"] = base["dragons"] - base["opp_dragons"]
base["barondiff"] = base["barons"] - base["opp_barons"]

wr = base[TARGET].mean()
print(f"sampel                    : {len(base)}")
print(f"win rate blue side        : {wr:.4f} ({int(base[TARGET].sum())}/{len(base)})")
print(f"baseline kelas mayoritas  : {max(wr, 1-wr):.4f}")
print(f"latih / uji (kronologis)  : {base['split'].isin(SPLIT_LATIH).sum()} / {base['split'].isin(SPLIT_UJI).sum()}")
print(f"reguler / playoff         : {(base['playoffs']==0).sum()} / {(base['playoffs']==1).sum()}")

sampel                    : 555
win rate blue side        : 0.5099 (283/555)
baseline kelas mayoritas  : 0.5099
latih / uji (kronologis)  : 349 / 206
reguler / playoff         : 476 / 79


---
## Langkah 2 — Konstruksi fitur dan snapshot

Kebijakan fitur pada versi revisi:

- **Model utama** (`objektif=False`, bawaan): lima fitur numerik bertimestamp.
  Himpunan fiturnya identik di keempat snapshot, sehingga selisih performa
  antar snapshot murni karena titik waktu.
- **Analisis sensitivitas** (`objektif=True`): fitur utama ditambah indikator
  objektif biner yang waktu perolehannya tidak dapat diverifikasi dari dataset.
- **Audit kebocoran** (`tambahan=[...]`): menambahkan `firstbaron` / `atakhandiff`
  hanya untuk mendemonstrasikan efek fitur yang tidak valid secara temporal.

In [95]:
FITUR_NUMERIK = ["golddiffat{t}", "xpdiffat{t}", "csdiffat{t}",
                 "killdiffat{t}", "assistdiffat{t}"]

# Hanya untuk analisis sensitivitas. Keempatnya tidak punya timestamp di dataset.
# Aturan masuk: sebuah indikator ikut di snapshot t hanya jika objektifnya sudah spawn
# sebelum menit t. Waktu spawn dari catatan patch 25.09 [25]: void grub 8:00 (6:00 sebelum
# patch 15.09), Rift Herald 15:00. First blood dan first dragon (spawn 5:00) selalu ikut.
# Waktu penutupan tidak didokumentasikan sehingga tidak dipakai.
FITUR_OBJEKTIF = {
    10: ["firstblood", "firstdragon", "grubdiff"],
    15: ["firstblood", "firstdragon", "grubdiff"],
    20: ["firstblood", "firstdragon", "grubdiff", "firstherald"],
    25: ["firstblood", "firstdragon", "grubdiff", "firstherald"],
}

# Hanya untuk uji kebocoran (demonstrasi), bukan model.
FITUR_AKHIR = ["dpm", "totalgold", "visionscore",
               "towerdiff", "dragondiff", "barondiff", "gamelength"]


def daftar_fitur(t, objektif=False, tambahan=()):
    f = [c.format(t=t) for c in FITUR_NUMERIK]
    if objektif:
        f += FITUR_OBJEKTIF[t]
    return f + list(tambahan)

In [96]:
def buat_snapshot(base, t, objektif=False, tambahan=()):
    """X, y, meta untuk snapshot menit ke-t.

    Bawaan: hanya fitur numerik bertimestamp (model utama).
    objektif=True : tambah indikator biner (analisis sensitivitas).
    tambahan=[..] : tambah kolom tertentu (audit kebocoran).

    Eksklusi memakai gamelength >= t*60, bukan dropna(). Pertandingan yang
    berakhir 24:00-24:59 tetap punya nilai di kolom menit 25, namun nilainya
    adalah keadaan akhir pertandingan.
    """
    fitur = daftar_fitur(t, objektif, tambahan)
    d = base[base["gamelength"] >= t * 60].dropna(subset=fitur + [TARGET])
    X = d[fitur].astype(float).reset_index(drop=True)
    y = d[TARGET].astype(int).reset_index(drop=True)
    meta = d[["gameid", "date", "split", "playoffs", "gamelength"]].reset_index(drop=True)
    return X, y, meta


def belah_kronologis(meta):
    return (meta["split"].isin(SPLIT_LATIH).to_numpy(),
            meta["split"].isin(SPLIT_UJI).to_numpy())


def belah_playoff(meta):
    return (meta["playoffs"] == 0).to_numpy(), (meta["playoffs"] == 1).to_numpy()

In [97]:
ringkasan = []
for t in SNAPSHOTS:
    X, y, meta = buat_snapshot(base, t)
    tr, te = belah_kronologis(meta)
    rg, po = belah_playoff(meta)
    ringkasan.append({
        "snapshot": t,
        "n": len(y),
        "dibuang": len(base) - len(y),
        "n_fitur": X.shape[1],
        "latih": int(tr.sum()),
        "uji": int(te.sum()),
        "reguler": int(rg.sum()),
        "playoff": int(po.sum()),
    })

pd.DataFrame(ringkasan)

,snapshot,n,dibuang,n_fitur,latih,uji,reguler,playoff
0,10,555,0,5,349,206,476,79
1,15,555,0,5,349,206,476,79
2,20,555,0,5,349,206,476,79
3,25,540,15,5,339,201,462,78


In [98]:
for t in SNAPSHOTS:
    print(f"menit {t:>2} utama    : {daftar_fitur(t)}")
    print(f"         ekstensi : {daftar_fitur(t, objektif=True)}")

menit 10 utama    : ['golddiffat10', 'xpdiffat10', 'csdiffat10', 'killdiffat10', 'assistdiffat10']
         ekstensi : ['golddiffat10', 'xpdiffat10', 'csdiffat10', 'killdiffat10', 'assistdiffat10', 'firstblood', 'firstdragon', 'grubdiff']
menit 15 utama    : ['golddiffat15', 'xpdiffat15', 'csdiffat15', 'killdiffat15', 'assistdiffat15']
         ekstensi : ['golddiffat15', 'xpdiffat15', 'csdiffat15', 'killdiffat15', 'assistdiffat15', 'firstblood', 'firstdragon', 'grubdiff']
menit 20 utama    : ['golddiffat20', 'xpdiffat20', 'csdiffat20', 'killdiffat20', 'assistdiffat20']
         ekstensi : ['golddiffat20', 'xpdiffat20', 'csdiffat20', 'killdiffat20', 'assistdiffat20', 'firstblood', 'firstdragon', 'grubdiff', 'firstherald']
menit 25 utama    : ['golddiffat25', 'xpdiffat25', 'csdiffat25', 'killdiffat25', 'assistdiffat25']
         ekstensi : ['golddiffat25', 'xpdiffat25', 'csdiffat25', 'killdiffat25', 'assistdiffat25', 'firstblood', 'firstdragon', 'grubdiff', 'firstherald']


In [99]:
# Deskriptif, termasuk indikator biner (objektif=True)
X, y, meta = buat_snapshot(base, 20, objektif=True)
X.describe().T[["count", "mean", "std", "min", "max"]].round(2)

,count,mean,std,min,max
golddiffat20,555.0,185.71,3515.26,-10871.0,10651.0
xpdiffat20,555.0,20.97,3090.13,-10440.0,8453.0
csdiffat20,555.0,-2.28,41.65,-130.0,122.0
killdiffat20,555.0,0.10,4.46,-20.0,15.0
assistdiffat20,555.0,0.24,10.46,-43.0,35.0
firstblood,555.0,0.50,0.50,0.0,1.0
firstdragon,555.0,0.39,0.49,0.0,1.0
grubdiff,555.0,0.75,2.89,-6.0,6.0
firstherald,555.0,0.62,0.49,0.0,1.0


In [100]:
gameid_bersama = set.intersection(*[set(buat_snapshot(base, t)[2]["gameid"]) for t in SNAPSHOTS])
print(f"gameid tersedia di seluruh snapshot: {len(gameid_bersama)}")

gameid tersedia di seluruh snapshot: 540


---
## Langkah 3 — Baseline pembanding

Dua pembanding yang harus dilampaui model:

1. **Kelas mayoritas** — selalu memprediksi kelas yang lebih sering muncul.
2. **Aturan tanda selisih gold** — tim yang unggul gold pada menit ke-t diprediksi menang.

Keduanya dilaporkan pada dua himpunan: seluruh sampel (deskriptif, untuk Tabel 2)
dan data uji saja (untuk perbandingan langsung dengan model di Langkah 4).

In [101]:
from scipy import stats


def wilson_ci(sukses, n, alpha=0.05):
    if n == 0:
        return (np.nan, np.nan)
    z = stats.norm.ppf(1 - alpha / 2)
    p = sukses / n
    penyebut = 1 + z**2 / n
    pusat = (p + z**2 / (2 * n)) / penyebut
    lebar = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / penyebut
    return (max(0.0, pusat - lebar), min(1.0, pusat + lebar))

In [102]:
def baseline_snapshot(base, t, hanya_uji=False):
    X, y, meta = buat_snapshot(base, t)
    if hanya_uji:
        _, te = belah_kronologis(meta)
        X, y = X[te].reset_index(drop=True), y[te].reset_index(drop=True)

    y = y.to_numpy()
    n = len(y)

    proporsi = y.mean()
    maj_rate = max(proporsi, 1 - proporsi)
    maj_benar = int(round(maj_rate * n))
    maj_lo, maj_hi = wilson_ci(maj_benar, n)

    prediksi = (X[f"golddiffat{t}"].to_numpy() > 0).astype(int)
    gold_benar = int((prediksi == y).sum())
    gold_acc = gold_benar / n
    gold_lo, gold_hi = wilson_ci(gold_benar, n)

    return {
        "snapshot": t,
        "n": n,
        "mayoritas": maj_rate,
        "mayoritas_lo": maj_lo,
        "mayoritas_hi": maj_hi,
        "tanda_gold": gold_acc,
        "tanda_gold_lo": gold_lo,
        "tanda_gold_hi": gold_hi,
    }

In [103]:
baseline_semua = pd.DataFrame([baseline_snapshot(base, t) for t in SNAPSHOTS])
baseline_uji = pd.DataFrame([baseline_snapshot(base, t, hanya_uji=True) for t in SNAPSHOTS])

print("seluruh sampel")
print(baseline_semua.round(4).to_string(index=False))
print("\ndata uji saja (Rounds 3-5)")
print(baseline_uji.round(4).to_string(index=False))

seluruh sampel
 snapshot   n  mayoritas  mayoritas_lo  mayoritas_hi  tanda_gold  tanda_gold_lo  tanda_gold_hi
       10 555     0.5099        0.4684        0.5513      0.6721         0.6319         0.7098
       15 555     0.5099        0.4684        0.5513      0.7153         0.6764         0.7513
       20 555     0.5099        0.4684        0.5513      0.7766         0.7401         0.8093
       25 540     0.5148        0.4727        0.5567      0.8259         0.7917         0.8556

data uji saja (Rounds 3-5)
 snapshot   n  mayoritas  mayoritas_lo  mayoritas_hi  tanda_gold  tanda_gold_lo  tanda_gold_hi
       10 206     0.5437        0.4755        0.6103      0.6845         0.6181         0.7441
       15 206     0.5437        0.4755        0.6103      0.6990         0.6332         0.7576
       20 206     0.5437        0.4755        0.6103      0.7427         0.6790         0.7976
       25 201     0.5473        0.4782        0.6145      0.8209         0.7620         0.8677


In [104]:
tabel2 = baseline_semua[["snapshot", "n", "mayoritas", "tanda_gold",
                         "tanda_gold_lo", "tanda_gold_hi"]].copy()
tabel2.columns = ["Snapshot", "Jumlah sampel", "Baseline kelas mayoritas",
                  "Baseline tanda selisih gold", "CI bawah", "CI atas"]
tabel2["Snapshot"] = "Menit " + tabel2["Snapshot"].astype(str)
tabel2.iloc[:, 2:] = (tabel2.iloc[:, 2:] * 100).round(1)

tabel2.to_csv("tabel2_baseline.csv", index=False)
tabel2

,Snapshot,Jumlah sampel,Baseline kelas mayoritas,Baseline tanda selisih gold,CI bawah,CI atas
0,Menit 10,555,51.0,67.2,63.2,71.0
1,Menit 15,555,51.0,71.5,67.6,75.1
2,Menit 20,555,51.0,77.7,74.0,80.9
3,Menit 25,540,51.5,82.6,79.2,85.6


In [105]:
ambang = baseline_uji.set_index("snapshot")["tanda_gold"].to_dict()
print("Ambang yang harus dilampaui model pada data uji:")
for t, v in ambang.items():
    print(f"  menit {t:<3} {v:.4f}")

Ambang yang harus dilampaui model pada data uji:
  menit 10  0.6845
  menit 15  0.6990
  menit 20  0.7427
  menit 25  0.8209


---
## Langkah 4 — Pemodelan dan evaluasi kronologis

Tiga model dilatih terpisah pada setiap snapshot, menghasilkan 12 kombinasi.
Fitur: **lima fitur numerik bertimestamp** (model utama).

**Alur setiap kombinasi:**

1. *Tuning* hyperparameter dengan 5-fold cross-validation **pada data latih saja**.
   Data uji tidak pernah disentuh pada tahap ini.
2. Model terbaik dilatih ulang pada beberapa nilai `seed`, lalu dievaluasi pada data uji.
   Rerata dan simpangan bakunya dilaporkan.
3. Metrik: accuracy (dengan selang kepercayaan Wilson), precision, recall,
   F1, AUC-ROC, dan Brier score.

In [106]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, brier_score_loss)

# MODE_CEPAT memangkas ruang pencarian dan jumlah seed. Berguna untuk memastikan
# seluruh notebook berjalan sebelum dijalankan penuh.
MODE_CEPAT = False

N_SEED = 1 if MODE_CEPAT else 10
CV_FOLD = 5

try:
    from xgboost import XGBClassifier
    ADA_XGBOOST = True
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    ADA_XGBOOST = False
    print("xgboost tidak terpasang -> memakai HistGradientBoostingClassifier "
          "sebagai pengganti. Pasang xgboost agar sesuai naskah.")

In [107]:
def daftar_model(seed=RANDOM_STATE):
    """Kembalikan {nama: (estimator, grid hyperparameter)}.

    Ruang pencarian sengaja ringkas. Dengan 349 data latih, grid yang terlalu
    lebar justru berisiko overfit pada tahap pemilihan hyperparameter itu sendiri.

    Catatan n_jobs: estimator diset n_jobs=1 dan paralelisasi ditaruh di
    GridSearchCV. Pada data sekecil ini, memparalelkan antar-fold jauh lebih
    efisien daripada antar-pohon, dan menghindari paralelisasi bersarang yang
    memicu UserWarning dari sklearn.
    """
    m = {}

    m["LogisticRegression"] = (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=5000, random_state=seed))]),
        {"clf__C": [0.01, 0.1, 1.0, 10.0]},
    )

    m["RandomForest"] = (
        RandomForestClassifier(random_state=seed, n_jobs=1),
        {"n_estimators": [300, 600],
         "max_depth": [4, 8, None],
         "min_samples_leaf": [1, 5],
         "max_features": ["sqrt"]},
    )

    if ADA_XGBOOST:
        m["XGBoost"] = (
            XGBClassifier(random_state=seed, eval_metric="logloss",
                          tree_method="hist", n_jobs=1),
            {"n_estimators": [200, 400],
             "max_depth": [2, 3, 5],
             "learning_rate": [0.03, 0.1],
             "subsample": [0.8, 1.0],
             "reg_lambda": [1.0, 5.0]},
        )
    else:
        m["XGBoost"] = (
            HistGradientBoostingClassifier(random_state=seed),
            {"max_iter": [200, 400],
             "max_depth": [2, 3, None],
             "learning_rate": [0.03, 0.1],
             "l2_regularization": [0.0, 1.0]},
        )

    if MODE_CEPAT:      # ambil satu nilai pertama dari tiap parameter
        m = {k: (est, {p: v[:1] for p, v in grid.items()}) for k, (est, grid) in m.items()}
    return m

In [108]:
def hitung_metrik(y_true, y_prob, ambang=0.5):
    """Metrik lengkap untuk satu model pada satu himpunan uji.

    Wilson CI dipakai, bukan pendekatan normal biasa, karena lebih andal pada
    ukuran sampel menengah dan proporsi yang jauh dari 0,5.
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= ambang).astype(int)

    n = len(y_true)
    benar = int((y_pred == y_true).sum())
    lo, hi = wilson_ci(benar, n)

    hasil = {
        "n": n,
        "accuracy": accuracy_score(y_true, y_pred),
        "acc_lo": lo,
        "acc_hi": hi,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "brier": brier_score_loss(y_true, y_prob),
    }
    # AUC tidak terdefinisi bila data uji hanya berisi satu kelas
    hasil["auc"] = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    return hasil

In [109]:
def cari_hyperparameter(estimator, grid, X_tr, y_tr, seed=RANDOM_STATE):
    """5-fold CV pada data latih. Metrik pemilihan: AUC."""
    if not grid:
        return clone(estimator).fit(X_tr, y_tr), {}
    cv = StratifiedKFold(n_splits=CV_FOLD, shuffle=True, random_state=seed)
    pencarian = GridSearchCV(estimator, grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    pencarian.fit(X_tr, y_tr)
    return pencarian.best_estimator_, pencarian.best_params_


def _ganti_seed(estimator, seed):
    """Ubah random_state, termasuk yang berada di dalam Pipeline."""
    ubah = {k: seed for k in estimator.get_params() if k.endswith("random_state")}
    if ubah:
        estimator.set_params(**ubah)


def latih_dan_evaluasi(nama, estimator, grid, X_tr, y_tr, X_te, y_te, n_seed=None):
    """Tuning sekali, lalu latih ulang pada beberapa seed.

    Mengembalikan (ringkasan metrik, probabilitas rata-rata, model terakhir).
    Probabilitas rata-rata dipakai untuk uji DeLong dan analisis subkelompok.
    """
    n_seed = n_seed or N_SEED
    terbaik, params = cari_hyperparameter(estimator, grid, X_tr, y_tr)

    per_seed, semua_prob = [], []
    for s in range(n_seed):
        est = clone(terbaik)
        _ganti_seed(est, RANDOM_STATE + s)
        est.fit(X_tr, y_tr)
        prob = est.predict_proba(X_te)[:, 1]
        semua_prob.append(prob)
        per_seed.append(hitung_metrik(y_te, prob))

    df = pd.DataFrame(per_seed)
    ringkas = {"model": nama, "params": str(params), "n_seed": n_seed}
    for kolom in ["n", "accuracy", "acc_lo", "acc_hi", "precision",
                  "recall", "f1", "auc", "brier"]:
        ringkas[kolom] = df[kolom].mean()
    ringkas["accuracy_sd"] = df["accuracy"].std(ddof=1)
    ringkas["auc_sd"] = df["auc"].std(ddof=1)

    return ringkas, np.mean(semua_prob, axis=0), est

In [110]:
%%time
# Eksperimen utama: 4 snapshot x 3 model, fitur numerik bertimestamp.
# `prediksi` menyimpan y dan probabilitas data uji untuk dipakai di Langkah 5.

hasil_utama = []
prediksi = {}

for t in SNAPSHOTS:
    X, y, meta = buat_snapshot(base, t)
    tr, te = belah_kronologis(meta)
    X_tr, y_tr = X[tr], y[tr]
    X_te, y_te = X[te].reset_index(drop=True), y[te].reset_index(drop=True)
    meta_te = meta[te].reset_index(drop=True)

    print(f"snapshot {t:>2} | latih {len(y_tr)} | uji {len(y_te)} | fitur {X.shape[1]}")

    for nama, (est, grid) in daftar_model().items():
        ringkas, prob, model = latih_dan_evaluasi(nama, est, grid,
                                                  X_tr, y_tr, X_te, y_te)
        ringkas["snapshot"] = t
        ringkas["n_fitur"] = X.shape[1]
        hasil_utama.append(ringkas)
        prediksi[(t, nama)] = {"y": y_te.to_numpy(), "prob": prob, "meta": meta_te}
        print(f"   {nama:<20} acc={ringkas['accuracy']:.4f}  auc={ringkas['auc']:.4f}")

hasil_utama = pd.DataFrame(hasil_utama)

snapshot 10 | latih 349 | uji 206 | fitur 5
   LogisticRegression   acc=0.6359  auc=0.7141
   RandomForest         acc=0.6816  auc=0.7024
   XGBoost              acc=0.6650  auc=0.6825
snapshot 15 | latih 349 | uji 206 | fitur 5
   LogisticRegression   acc=0.6748  auc=0.7642
   RandomForest         acc=0.6650  auc=0.7531
   XGBoost              acc=0.6650  auc=0.7254
snapshot 20 | latih 349 | uji 206 | fitur 5
   LogisticRegression   acc=0.7379  auc=0.8423
   RandomForest         acc=0.7602  auc=0.8477
   XGBoost              acc=0.7456  auc=0.8439
snapshot 25 | latih 339 | uji 201 | fitur 5
   LogisticRegression   acc=0.8159  auc=0.9065
   RandomForest         acc=0.8164  auc=0.9049
   XGBoost              acc=0.8219  auc=0.9050
CPU times: total: 1min 30s
Wall time: 3min 52s


In [111]:
kolom_tampil = ["snapshot", "model", "n", "accuracy", "acc_lo", "acc_hi",
                "accuracy_sd", "f1", "auc", "auc_sd", "brier"]
tabel_utama = hasil_utama[kolom_tampil].sort_values(["snapshot", "model"])
tabel_utama.round(4)

,snapshot,model,n,accuracy,acc_lo,acc_hi,accuracy_sd,f1,auc,auc_sd,brier
0,10,LogisticRegression,206.0,0.6359,0.5683,0.6986,0.0000,0.6479,0.7141,0.0000,0.2200
1,10,RandomForest,206.0,0.6816,0.6151,0.7413,0.0083,0.6940,0.7024,0.0030,0.2282
2,10,XGBoost,206.0,0.6650,0.5981,0.7260,0.0000,0.6761,0.6825,0.0000,0.2294
3,15,LogisticRegression,206.0,0.6748,0.6081,0.7350,0.0000,0.6854,0.7642,0.0000,0.2016
4,15,RandomForest,206.0,0.6650,0.5981,0.7260,0.0040,0.6782,0.7531,0.0012,0.2132
5,15,XGBoost,206.0,0.6650,0.5981,0.7260,0.0000,0.6820,0.7254,0.0000,0.2313
6,20,LogisticRegression,206.0,0.7379,0.6738,0.7932,0.0000,0.7245,0.8423,0.0000,0.1652
7,20,RandomForest,206.0,0.7602,0.6975,0.8134,0.0061,0.7461,0.8477,0.0007,0.1668
8,20,XGBoost,206.0,0.7456,0.6820,0.8002,0.0057,0.7335,0.8439,0.0013,0.1712
9,25,LogisticRegression,201.0,0.8159,0.7566,0.8634,0.0000,0.8063,0.9065,0.0000,0.1342


In [112]:
# Hyperparameter terpilih per snapshot dan model (bahan Tabel III naskah).
# Pada versi revisi himpunan fitur berubah, jadi nilai terpilih perlu dibaca ulang.
print(hasil_utama[["snapshot", "model", "params"]]
      .sort_values(["model", "snapshot"]).to_string(index=False))

 snapshot              model                                                                                            params
       10 LogisticRegression                                                                                   {'clf__C': 0.1}
       15 LogisticRegression                                                                                  {'clf__C': 0.01}
       20 LogisticRegression                                                                                   {'clf__C': 0.1}
       25 LogisticRegression                                                                                  {'clf__C': 0.01}
       10       RandomForest              {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 600}
       15       RandomForest              {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 600}
       20       RandomForest           {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_es

In [113]:
# Apakah model benar-benar melampaui baseline aturan tanda gold pada data uji?
# Model dinyatakan unggul hanya bila batas bawah CI-nya di atas nilai baseline.

banding = tabel_utama.copy()
banding["baseline_uji"] = banding["snapshot"].map(ambang)
banding["selisih"] = banding["accuracy"] - banding["baseline_uji"]
banding["melampaui_ci"] = banding["acc_lo"] > banding["baseline_uji"]

banding[["snapshot", "model", "accuracy", "acc_lo", "baseline_uji",
         "selisih", "melampaui_ci"]].round(4)

,snapshot,model,accuracy,acc_lo,baseline_uji,selisih,melampaui_ci
0,10,LogisticRegression,0.6359,0.5683,0.6845,-0.0485,False
1,10,RandomForest,0.6816,0.6151,0.6845,-0.0029,False
2,10,XGBoost,0.6650,0.5981,0.6845,-0.0194,False
3,15,LogisticRegression,0.6748,0.6081,0.6990,-0.0243,False
4,15,RandomForest,0.6650,0.5981,0.6990,-0.0340,False
5,15,XGBoost,0.6650,0.5981,0.6990,-0.0340,False
6,20,LogisticRegression,0.7379,0.6738,0.7427,-0.0049,False
7,20,RandomForest,0.7602,0.6975,0.7427,0.0175,False
8,20,XGBoost,0.7456,0.6820,0.7427,0.0029,False
9,25,LogisticRegression,0.8159,0.7566,0.8209,-0.0050,False


---
## Langkah 5 — Uji lanjutan

Enam analisis:

1. **Uji DeLong** — apakah selisih AUC antar model bermakna secara statistik.
2. **Uji baseline probabilistik** -  
3. **Kriteria keandalan** — menjawab pada menit berapa model mulai andal.
4. **Varian ekstensi (sensitivitas)** — menambah indikator objektif biner ke model utama.
5. **Sensitivitas subset** — seluruh snapshot pada subset sampel yang sama.
6. **Subkelompok playoff** (termasuk uji proporsi dan Fisher).
7. **Uji kebocoran data** — fitur akhir pertandingan vs snapshot menit 25.

In [114]:
# --- Uji DeLong ---------------------------------------------------------
# Selisih AUC dua model pada data uji yang SAMA bersifat berkorelasi, sehingga
# tidak boleh diuji dengan uji dua sampel biasa. Uji DeLong memperhitungkan
# korelasi tersebut. Implementasi di bawah memakai pendekatan mid-rank yang
# menurunkan kompleksitas dari kuadratik menjadi linearitmik.

def _midrank(x):
    urut = np.argsort(x)
    z = x[urut]
    n = len(x)
    t = np.empty(n, dtype=float)
    i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1     # nilai kembar berbagi rank yang sama
        i = j
    keluar = np.empty(n, dtype=float)
    keluar[urut] = t
    return keluar


def _delong_cov(pred_terurut, n_pos):
    """pred_terurut: (k, N), n_pos kolom pertama adalah kelas positif."""
    m, n = n_pos, pred_terurut.shape[1] - n_pos
    pos, neg = pred_terurut[:, :m], pred_terurut[:, m:]
    k = pred_terurut.shape[0]

    tx = np.empty((k, m)); ty = np.empty((k, n)); tz = np.empty((k, m + n))
    for r in range(k):
        tx[r] = _midrank(pos[r])
        ty[r] = _midrank(neg[r])
        tz[r] = _midrank(pred_terurut[r])

    auc = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.atleast_2d(np.cov(v01, ddof=1)) / m + np.atleast_2d(np.cov(v10, ddof=1)) / n
    return auc, cov


def uji_delong(y_true, prob_a, prob_b):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return {"auc_a": np.nan, "auc_b": np.nan, "selisih": np.nan,
                "z": np.nan, "p": np.nan}

    urut = np.argsort(-y_true, kind="mergesort")          # positif lebih dulu
    pred = np.vstack([np.asarray(prob_a, float), np.asarray(prob_b, float)])[:, urut]
    auc, cov = _delong_cov(pred, int(y_true.sum()))

    kontras = np.array([[1.0, -1.0]])
    var = float((kontras @ cov @ kontras.T).item())     # .item() wajib di numpy 2.x
    selisih = float(auc[0] - auc[1])

    if var <= 0:
        z, p = (0.0, 1.0)
    else:
        z = selisih / np.sqrt(var)
        p = 2 * (1 - stats.norm.cdf(abs(z)))
    return {"auc_a": float(auc[0]), "auc_b": float(auc[1]),
            "selisih": selisih, "z": float(z), "p": float(p)}

In [115]:
nama_model = sorted({k[1] for k in prediksi})
baris = []
for t in SNAPSHOTS:
    for i in range(len(nama_model)):
        for j in range(i + 1, len(nama_model)):
            a, b = nama_model[i], nama_model[j]
            r = uji_delong(prediksi[(t, a)]["y"],
                           prediksi[(t, a)]["prob"],
                           prediksi[(t, b)]["prob"])
            r.update({"snapshot": t, "model_a": a, "model_b": b})
            baris.append(r)

tabel_delong = pd.DataFrame(baris)[["snapshot", "model_a", "model_b",
                                    "auc_a", "auc_b", "selisih", "z", "p"]]
tabel_delong["signifikan"] = tabel_delong["p"] < 0.05
# koreksi Bonferroni untuk 12 perbandingan (naskah melaporkan ambang 0,05/12)
tabel_delong["signifikan_bonferroni"] = tabel_delong["p"] < 0.05 / len(tabel_delong)
tabel_delong.round(4)

,snapshot,model_a,model_b,auc_a,auc_b,selisih,z,p,signifikan,signifikan_bonferroni
0,10,LogisticRegression,RandomForest,0.7141,0.7028,0.0113,0.7207,0.4711,False,False
1,10,LogisticRegression,XGBoost,0.7141,0.6825,0.0316,1.5105,0.1309,False,False
2,10,RandomForest,XGBoost,0.7028,0.6825,0.0203,1.7881,0.0738,False,False
3,15,LogisticRegression,RandomForest,0.7642,0.7530,0.0111,1.0494,0.2940,False,False
4,15,LogisticRegression,XGBoost,0.7642,0.7254,0.0387,2.0066,0.0448,True,False
5,15,RandomForest,XGBoost,0.7530,0.7254,0.0276,2.0781,0.0377,True,False
6,20,LogisticRegression,RandomForest,0.8423,0.8478,-0.0055,-0.7396,0.4596,False,False
7,20,LogisticRegression,XGBoost,0.8423,0.8439,-0.0016,-0.2619,0.7934,False,False
8,20,RandomForest,XGBoost,0.8478,0.8439,0.0039,0.9602,0.3369,False,False
9,25,LogisticRegression,RandomForest,0.9065,0.9053,0.0012,0.2201,0.8258,False,False


In [116]:
# --- Baseline probabilistik: regresi logistik satu fitur (selisih gold) ---
# Pembanding yang sah untuk Brier dan AUC. Aturan tanda gold tidak menghasilkan
# probabilitas, jadi tidak bisa dipakai untuk itu.
import os
baris_bp, prob_bp = [], {}
for t in SNAPSHOTS:
    X, y, meta = buat_snapshot(base, t)
    tr, te = belah_kronologis(meta)
    kol = f"golddiffat{t}"
    m = Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=5000))])
    m.fit(X.loc[tr, [kol]], y[tr])
    prob = m.predict_proba(X.loc[te, [kol]])[:, 1]
    y_te = y[te].reset_index(drop=True)
    prob_bp[t] = prob
    r = hitung_metrik(y_te, prob)
    r["snapshot"] = t
    baris_bp.append(r)

tabel_bp = pd.DataFrame(baris_bp)[["snapshot", "accuracy", "auc", "brier"]]

gabung = (tabel_utama[["snapshot", "model", "accuracy", "auc", "brier"]]
          .merge(tabel_bp, on="snapshot", suffixes=("", "_gold_saja")))
gabung["selisih_brier"] = gabung["brier"] - gabung["brier_gold_saja"]  # negatif = model lebih baik
gabung["selisih_auc"] = gabung["auc"] - gabung["auc_gold_saja"]

# uji DeLong: model penuh vs regresi logistik gold saja
p_delong = []
for _, r in gabung.iterrows():
    d = prediksi[(r["snapshot"], r["model"])]
    p_delong.append(uji_delong(d["y"], d["prob"], prob_bp[r["snapshot"]])["p"])
gabung["p_delong_vs_gold_saja"] = p_delong

os.makedirs("outputs", exist_ok=True)
gabung.to_csv("outputs/baseline_probabilistik.csv", index=False)
gabung.round(4)

,snapshot,model,accuracy,auc,brier,accuracy_gold_saja,auc_gold_saja,brier_gold_saja,selisih_brier,selisih_auc,p_delong_vs_gold_saja
0,10,LogisticRegression,0.6359,0.7141,0.2200,0.6553,0.7114,0.2213,-0.0014,0.0027,0.8301
1,10,RandomForest,0.6816,0.7024,0.2282,0.6553,0.7114,0.2213,0.0069,-0.0090,0.6732
2,10,XGBoost,0.6650,0.6825,0.2294,0.6553,0.7114,0.2213,0.0080,-0.0289,0.2389
3,15,LogisticRegression,0.6748,0.7642,0.2016,0.7233,0.7791,0.1944,0.0073,-0.0149,0.1612
4,15,RandomForest,0.6650,0.7531,0.2132,0.7233,0.7791,0.1944,0.0188,-0.0259,0.0670
5,15,XGBoost,0.6650,0.7254,0.2313,0.7233,0.7791,0.1944,0.0370,-0.0536,0.0119
6,20,LogisticRegression,0.7379,0.8423,0.1652,0.7427,0.8349,0.1703,-0.0051,0.0074,0.4382
7,20,RandomForest,0.7602,0.8477,0.1668,0.7427,0.8349,0.1703,-0.0035,0.0128,0.1848
8,20,XGBoost,0.7456,0.8439,0.1712,0.7427,0.8349,0.1703,0.0010,0.0090,0.3557
9,25,LogisticRegression,0.8159,0.9065,0.1342,0.8209,0.8974,0.1317,0.0026,0.0091,0.2174


In [117]:
# --- Kriteria keandalan (ditetapkan di awal, lihat Metode) ---------------
# Snapshot paling awal yang memenuhi:
#   (i)  AUC >= 0,85
#   (ii) kenaikan AUC ke snapshot berikutnya < 0,02  (performa sudah mendatar)

AUC_MIN = 0.85
DELTA_DATAR = 0.02


def snapshot_andal(auc_per_snapshot, snapshots=SNAPSHOTS):
    for i, t in enumerate(snapshots):
        auc = auc_per_snapshot.get(t, np.nan)
        if np.isnan(auc) or auc < AUC_MIN:
            continue
        if i == len(snapshots) - 1:          # snapshot terakhir: hanya syarat (i)
            return t
        berikut = auc_per_snapshot.get(snapshots[i + 1], np.nan)
        if np.isnan(berikut) or (berikut - auc) < DELTA_DATAR:
            return t
    return None


baris = []
for m in hasil_utama["model"].unique():
    sub = hasil_utama[hasil_utama["model"] == m]
    peta = dict(zip(sub["snapshot"], sub["auc"]))
    baris.append({"model": m, "snapshot_andal": snapshot_andal(peta),
                  **{f"auc_t{t}": round(peta.get(t, np.nan), 4) for t in SNAPSHOTS}})

tabel_keandalan = pd.DataFrame(baris)
print(tabel_keandalan.to_string(index=False))

# Diagnostik: apakah syarat (ii) pernah benar-benar menentukan hasil?
# Kalau AUC masih menanjak tajam sampai snapshot terakhir, syarat mendatar
# tidak pernah terpakai dan jawaban RQ-2 sepenuhnya ditentukan oleh ambang
# AUC. Ini perlu dinyatakan di pembahasan, bukan disembunyikan.
print("\nkenaikan AUC antar snapshot:")
for m in hasil_utama["model"].unique():
    sub = hasil_utama[hasil_utama["model"] == m].sort_values("snapshot")
    delta = sub["auc"].diff().dropna()
    teks = "  ".join(f"{a}->{b}: {d:+.4f}"
                     for a, b, d in zip(SNAPSHOTS[:-1], SNAPSHOTS[1:], delta))
    mendatar = (delta < DELTA_DATAR).any()
    print(f"  {m:<20} {teks}   pernah mendatar: {mendatar}")

tabel_keandalan

             model  snapshot_andal  auc_t10  auc_t15  auc_t20  auc_t25
LogisticRegression              25   0.7141   0.7642   0.8423   0.9065
      RandomForest              25   0.7024   0.7531   0.8477   0.9049
           XGBoost              25   0.6825   0.7254   0.8439   0.9050

kenaikan AUC antar snapshot:
  LogisticRegression   10->15: +0.0501  15->20: +0.0782  20->25: +0.0642   pernah mendatar: False
  RandomForest         10->15: +0.0507  15->20: +0.0946  20->25: +0.0572   pernah mendatar: False
  XGBoost              10->15: +0.0429  15->20: +0.1185  20->25: +0.0611   pernah mendatar: False


,model,snapshot_andal,auc_t10,auc_t15,auc_t20,auc_t25
0,LogisticRegression,25,0.7141,0.7642,0.8423,0.9065
1,RandomForest,25,0.7024,0.7531,0.8477,0.9049
2,XGBoost,25,0.6825,0.7254,0.8439,0.9050


In [118]:
%%time
# --- Analisis sensitivitas: menambah indikator objektif biner -----------
# Fitur utama + first blood, first dragon, void grub (menit 20, 25) dan
# first herald (menit 25). Waktu perolehan keempatnya TIDAK dapat diverifikasi
# dari dataset, sehingga hasil di sini hanya untuk menguji apakah kesimpulan
# bergantung pada pengeluaran fitur tersebut dari model utama.

hasil_ekstensi = []
for t in SNAPSHOTS:
    X, y, meta = buat_snapshot(base, t, objektif=True)
    tr, te = belah_kronologis(meta)
    for nama, (est, grid) in daftar_model().items():
        ringkas, _, _ = latih_dan_evaluasi(nama, est, grid,
                                           X[tr], y[tr],
                                           X[te].reset_index(drop=True),
                                           y[te].reset_index(drop=True))
        ringkas.update({"snapshot": t, "varian": "dengan_indikator_biner"})
        hasil_ekstensi.append(ringkas)

hasil_ekstensi = pd.DataFrame(hasil_ekstensi)
hasil_ekstensi[["snapshot", "model", "n", "accuracy", "auc", "brier"]].round(4)

CPU times: total: 1min 43s
Wall time: 4min 23s


,snapshot,model,n,accuracy,auc,brier
0,10,LogisticRegression,206.0,0.6262,0.7185,0.2225
1,10,RandomForest,206.0,0.6549,0.7119,0.2229
2,10,XGBoost,206.0,0.6330,0.6919,0.2272
3,15,LogisticRegression,206.0,0.6650,0.7680,0.2003
4,15,RandomForest,206.0,0.6684,0.7586,0.2080
5,15,XGBoost,206.0,0.6714,0.7341,0.2244
6,20,LogisticRegression,206.0,0.7427,0.8437,0.1644
7,20,RandomForest,206.0,0.7403,0.8441,0.1664
8,20,XGBoost,206.0,0.7447,0.8421,0.1724
9,25,LogisticRegression,201.0,0.8259,0.9075,0.1331


In [119]:
# Perbandingan model utama vs ekstensi (pengganti Tabel IV naskah).
# selisih > 0 berarti indikator biner menaikkan metrik.
tabel4 = (hasil_utama[["snapshot", "model", "accuracy", "auc"]]
          .merge(hasil_ekstensi[["snapshot", "model", "accuracy", "auc"]],
                 on=["snapshot", "model"], suffixes=("_utama", "_ekstensi")))
tabel4["selisih_acc"] = tabel4["accuracy_ekstensi"] - tabel4["accuracy_utama"]
tabel4["selisih_auc"] = tabel4["auc_ekstensi"] - tabel4["auc_utama"]
tabel4.sort_values(["snapshot", "model"]).round(4)

,snapshot,model,accuracy_utama,auc_utama,accuracy_ekstensi,auc_ekstensi,selisih_acc,selisih_auc
0,10,LogisticRegression,0.6359,0.7141,0.6262,0.7185,-0.0097,0.0044
1,10,RandomForest,0.6816,0.7024,0.6549,0.7119,-0.0267,0.0095
2,10,XGBoost,0.6650,0.6825,0.6330,0.6919,-0.0320,0.0094
3,15,LogisticRegression,0.6748,0.7642,0.6650,0.7680,-0.0097,0.0039
4,15,RandomForest,0.6650,0.7531,0.6684,0.7586,0.0034,0.0054
5,15,XGBoost,0.6650,0.7254,0.6714,0.7341,0.0063,0.0087
6,20,LogisticRegression,0.7379,0.8423,0.7427,0.8437,0.0049,0.0013
7,20,RandomForest,0.7602,0.8477,0.7403,0.8441,-0.0199,-0.0036
8,20,XGBoost,0.7456,0.8439,0.7447,0.8421,-0.0010,-0.0018
9,25,LogisticRegression,0.8159,0.9065,0.8259,0.9075,0.0100,0.0010


In [120]:
%%time
# --- Analisis sensitivitas subset ---------------------------------------
# Snapshot menit 25 kehilangan pertandingan berdurasi pendek, yang justru
# pertandingan paling timpang. Agar perbandingan antar snapshot setara,
# seluruh snapshot dihitung ulang pada subset gameid yang sama.

hasil_sensitivitas = []
for t in SNAPSHOTS:
    X, y, meta = buat_snapshot(base, t)
    pilih = meta["gameid"].isin(gameid_bersama).to_numpy()
    X, y, meta = (X[pilih].reset_index(drop=True),
                  y[pilih].reset_index(drop=True),
                  meta[pilih].reset_index(drop=True))
    tr, te = belah_kronologis(meta)
    for nama, (est, grid) in daftar_model().items():
        ringkas, _, _ = latih_dan_evaluasi(nama, est, grid,
                                           X[tr], y[tr],
                                           X[te].reset_index(drop=True),
                                           y[te].reset_index(drop=True))
        ringkas.update({"snapshot": t, "varian": "subset_bersama"})
        hasil_sensitivitas.append(ringkas)

hasil_sensitivitas = pd.DataFrame(hasil_sensitivitas)
hasil_sensitivitas[["snapshot", "model", "n", "accuracy", "auc", "brier"]].round(4)

CPU times: total: 1min 30s
Wall time: 4min 7s


,snapshot,model,n,accuracy,auc,brier
0,10,LogisticRegression,201.0,0.6269,0.7008,0.2254
1,10,RandomForest,201.0,0.6706,0.6889,0.2335
2,10,XGBoost,201.0,0.6473,0.6728,0.2357
3,15,LogisticRegression,201.0,0.6617,0.7530,0.2068
4,15,RandomForest,201.0,0.6612,0.7432,0.2179
5,15,XGBoost,201.0,0.6557,0.7248,0.2310
6,20,LogisticRegression,201.0,0.7463,0.8319,0.1747
7,20,RandomForest,201.0,0.7373,0.8384,0.1718
8,20,XGBoost,201.0,0.7403,0.8364,0.1762
9,25,LogisticRegression,201.0,0.8159,0.9065,0.1342


In [121]:
# --- Subkelompok playoff ------------------------------------------------
# Bukan skema pelatihan terpisah. Model yang sama dari Langkah 4 dievaluasi
# ulang pada dua subset data uji: musim reguler dan playoff.
#
# Alasannya: melatih pada seluruh musim reguler lalu menguji pada seluruh
# playoff akan melanggar kronologi, karena playoff Rounds 1-2 (Juni) akan
# diprediksi model yang sudah melihat pertandingan Juli-Agustus.
#
# Perhatikan daya statistik: dengan n=61, selang kepercayaan akurasi mencapai
# sekitar +/-11 poin. Hasil di sini adalah indikasi, bukan kesimpulan.

baris = []
for (t, nama), d in prediksi.items():
    meta_te = d["meta"]
    for label, mask in [("semua", np.ones(len(meta_te), bool)),
                        ("reguler", (meta_te["playoffs"] == 0).to_numpy()),
                        ("playoff", (meta_te["playoffs"] == 1).to_numpy())]:
        if mask.sum() == 0:
            continue
        m = hitung_metrik(d["y"][mask], d["prob"][mask])
        m.update({"snapshot": t, "model": nama, "subkelompok": label})
        baris.append(m)

tabel_subkelompok = (pd.DataFrame(baris)
                     [["snapshot", "model", "subkelompok", "n",
                       "accuracy", "acc_lo", "acc_hi", "auc", "brier"]]
                     .sort_values(["snapshot", "model", "subkelompok"]))
tabel_subkelompok.round(4)

,snapshot,model,subkelompok,n,accuracy,acc_lo,acc_hi,auc,brier
2,10,LogisticRegression,playoff,61,0.6066,0.4812,0.7193,0.6983,0.2299
1,10,LogisticRegression,reguler,145,0.6483,0.5676,0.7213,0.7233,0.2158
0,10,LogisticRegression,semua,206,0.6359,0.5683,0.6986,0.7141,0.2200
5,10,RandomForest,playoff,61,0.6557,0.5305,0.7625,0.7004,0.2315
4,10,RandomForest,reguler,145,0.6966,0.6174,0.7655,0.7074,0.2267
3,10,RandomForest,semua,206,0.6845,0.6181,0.7441,0.7028,0.2281
8,10,XGBoost,playoff,61,0.6721,0.5472,0.7766,0.6977,0.2261
7,10,XGBoost,reguler,145,0.6621,0.5818,0.7340,0.6794,0.2307
6,10,XGBoost,semua,206,0.6650,0.5981,0.7260,0.6825,0.2294
11,15,LogisticRegression,playoff,61,0.7049,0.5811,0.8044,0.8083,0.1867


In [122]:
# Selisih akurasi reguler vs playoff, disandingkan dengan lebar CI playoff.
#
# Kolom `layak_dibahas` hanyalah penyaring kasar, BUKAN uji statistik: ia
# menandai kasus yang selisihnya melebihi separuh lebar CI playoff. Selisih
# di bawah itu jangan diklaim sebagai temuan sama sekali. Bahkan yang lolos
# penyaring pun sebaiknya ditulis sebagai indikasi, karena n=61.

piv = tabel_subkelompok.pivot_table(index=["snapshot", "model"],
                                    columns="subkelompok",
                                    values=["accuracy", "acc_lo", "acc_hi"])
ringkas_sub = pd.DataFrame({
    "acc_reguler": piv[("accuracy", "reguler")],
    "acc_playoff": piv[("accuracy", "playoff")],
    "selisih": piv[("accuracy", "playoff")] - piv[("accuracy", "reguler")],
    "lebar_ci_playoff": piv[("acc_hi", "playoff")] - piv[("acc_lo", "playoff")],
}).round(4)
ringkas_sub["layak_dibahas"] = (ringkas_sub["selisih"].abs()
                                > ringkas_sub["lebar_ci_playoff"] / 2)
ringkas_sub

acc_reguler  acc_playoff  selisih  lebar_ci_playoff  layak_dibahas
snapshot model                                                                                 
10       LogisticRegression       0.6483       0.6066  -0.0417            0.2381          False
         RandomForest             0.6966       0.6557  -0.0408            0.2320          False
         XGBoost                  0.6621       0.6721   0.0101            0.2294          False
15       LogisticRegression       0.6621       0.7049   0.0428            0.2233          False
         RandomForest             0.6483       0.6885   0.0402            0.2265          False
         XGBoost                  0.6552       0.6885   0.0334            0.2265          False
20       LogisticRegression       0.7172       0.7869   0.0696            0.2022          False
         RandomForest             0.7379       0.8197   0.0817            0.1910          False
         XGBoost                  0.7379       0.7705   0.0326            0.2072          False
25       LogisticRegression       0.7730       0.9167   0.1436            0.1446           True
         RandomForest             0.7660       0.9333   0.1674            0.1330           True
         XGBoost                  0.7801       0.9333   0.1532            0.1330           True

In [123]:
# Uji proporsi dua sampel dan uji eksak Fisher: akurasi reguler vs playoff.
# Naskah melaporkan kedua uji ini untuk snapshot menit 25 (dan menit 20 sebagai
# pembanding). Tanpa koreksi: ini temuan eksploratori, bukan konfirmatori.
from scipy.stats import fisher_exact


def uji_proporsi(b1, n1, b2, n2):
    p = (b1 + b2) / (n1 + n2)
    se = np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    z = (b2 / n2 - b1 / n1) / se
    return z, 2 * (1 - stats.norm.cdf(abs(z)))


baris = []
for (t, nama), d in prediksi.items():
    benar = ((d["prob"] >= 0.5).astype(int) == d["y"])
    po = (d["meta"]["playoffs"] == 1).to_numpy()
    b_po, n_po = int(benar[po].sum()), int(po.sum())
    b_rg, n_rg = int(benar[~po].sum()), int((~po).sum())
    z, p = uji_proporsi(b_rg, n_rg, b_po, n_po)
    _, p_fisher = fisher_exact([[b_po, n_po - b_po], [b_rg, n_rg - b_rg]])
    baris.append({"snapshot": t, "model": nama,
                  "acc_reguler": b_rg / n_rg, "acc_playoff": b_po / n_po,
                  "selisih": b_po / n_po - b_rg / n_rg,
                  "n_reguler": n_rg, "n_playoff": n_po,
                  "z": z, "p_proporsi": p, "p_fisher": p_fisher})

tabel_uji_playoff = pd.DataFrame(baris).sort_values(["snapshot", "model"])
tabel_uji_playoff.round(4)

,snapshot,model,acc_reguler,acc_playoff,selisih,n_reguler,n_playoff,z,p_proporsi,p_fisher
0,10,LogisticRegression,0.6483,0.6066,-0.0417,145,61,-0.5681,0.5699,0.6349
1,10,RandomForest,0.6966,0.6557,-0.0408,145,61,-0.5755,0.5650,0.6231
2,10,XGBoost,0.6621,0.6721,0.0101,145,61,0.1397,0.8889,1.0000
3,15,LogisticRegression,0.6621,0.7049,0.0428,145,61,0.5993,0.5489,0.6262
4,15,RandomForest,0.6483,0.6885,0.0402,145,61,0.5568,0.5776,0.6310
5,15,XGBoost,0.6552,0.6885,0.0334,145,61,0.4630,0.6433,0.7468
6,20,LogisticRegression,0.7172,0.7869,0.0696,145,61,1.0376,0.2994,0.3858
7,20,RandomForest,0.7379,0.8197,0.0817,145,61,1.2580,0.2084,0.2820
8,20,XGBoost,0.7379,0.7705,0.0326,145,61,0.4912,0.6233,0.7262
9,25,LogisticRegression,0.7730,0.9167,0.1436,141,60,2.4042,0.0162,0.0168


In [124]:
%%time
# --- Uji kebocoran data -------------------------------------------------
# Fitur akhir pertandingan TIDAK diusulkan sebagai model. Ini demonstrasi.
#
# Kriteria kebocoran (ditetapkan di awal):
#   (i)  nilainya baru diketahui setelah pertandingan berakhir, sehingga
#        tidak tersedia pada saat prediksi dilakukan
#   (ii) performanya tidak dapat dicapai oleh model snapshot mana pun
#
# Syarat (i) bersifat argumentatif, bukan statistik. Akurasi tinggi saja bukan
# bukti kebocoran; yang menentukan adalah ketersediaan fitur pada saat prediksi.

AMBANG_KEBOCORAN_AUC = 0.98

d_akhir = base.dropna(subset=FITUR_AKHIR + [TARGET])
X_akhir = d_akhir[FITUR_AKHIR].astype(float).reset_index(drop=True)
y_akhir = d_akhir[TARGET].astype(int).reset_index(drop=True)
meta_akhir = d_akhir[["gameid", "split", "playoffs"]].reset_index(drop=True)
tr_a, te_a = belah_kronologis(meta_akhir)

hasil_kebocoran = []
for nama, (est, grid) in daftar_model().items():
    ringkas, _, _ = latih_dan_evaluasi(nama, est, grid,
                                       X_akhir[tr_a], y_akhir[tr_a],
                                       X_akhir[te_a].reset_index(drop=True),
                                       y_akhir[te_a].reset_index(drop=True))
    ringkas.update({"himpunan_fitur": "akhir_pertandingan",
                    "tersedia_saat_prediksi": False})
    hasil_kebocoran.append(ringkas)

# pembanding yang sah: model snapshot menit 25 (fitur numerik bertimestamp)
for _, r in hasil_utama[hasil_utama["snapshot"] == 25].iterrows():
    hasil_kebocoran.append({**r.to_dict(),
                            "himpunan_fitur": "snapshot_25",
                            "tersedia_saat_prediksi": True})

hasil_kebocoran = pd.DataFrame(hasil_kebocoran)
hasil_kebocoran["indikasi_bocor"] = (
    (~hasil_kebocoran["tersedia_saat_prediksi"]) &
    (hasil_kebocoran["auc"] >= AMBANG_KEBOCORAN_AUC)
)
hasil_kebocoran[["himpunan_fitur", "model", "n", "accuracy", "auc",
                 "tersedia_saat_prediksi", "indikasi_bocor"]].round(4)

CPU times: total: 7.27 s
Wall time: 14.6 s


,himpunan_fitur,model,n,accuracy,auc,tersedia_saat_prediksi,indikasi_bocor
0,akhir_pertandingan,LogisticRegression,206.0,0.9854,0.9992,False,True
1,akhir_pertandingan,RandomForest,206.0,0.9757,0.9981,False,True
2,akhir_pertandingan,XGBoost,206.0,0.9714,0.9960,False,True
3,snapshot_25,LogisticRegression,201.0,0.8159,0.9065,True,False
4,snapshot_25,RandomForest,201.0,0.8164,0.9049,True,False
5,snapshot_25,XGBoost,201.0,0.8219,0.9050,True,False


---
## Langkah 6 — Kepentingan fitur

SHAP dipakai sebagai metrik utama untuk **ketiga** model supaya kepentingan
fitur dibandingkan pada skala yang sama.

Kenapa tidak memakai metrik bawaan masing-masing model? Karena koefisien
regresi, *Gini importance*, dan SHAP tidak sebanding satu sama lain. *Gini
importance* juga diketahui bias terhadap fitur kontinu berkardinalitas tinggi,
sehingga akan mengunggulkan selisih gold di atas fitur biner secara artifisial
terlepas dari kenyataannya.

*Permutation importance* pada data uji dipakai sebagai pemeriksaan silang,
sedangkan metrik bawaan tetap dilaporkan sebagai pelengkap.

Dihitung untuk dua varian: **utama** (fitur numerik saja) dan **ekstensi**
(ditambah indikator biner). Perbandingan numerik vs biner (naskah, Fig. 10)
bersumber dari varian ekstensi.

In [125]:
from sklearn.inspection import permutation_importance

try:
    import shap
    ADA_SHAP = True
except ImportError:
    ADA_SHAP = False
    print("shap tidak terpasang -> kolom shap akan kosong. "
          "Pasang dengan: pip install shap")


def nilai_shap(model, nama, X_tr, X_te):
    """Rata-rata |SHAP value| per fitur pada data uji."""
    if not ADA_SHAP:
        return None

    inti = model.named_steps["clf"] if isinstance(model, Pipeline) else model

    if isinstance(model, Pipeline):
        # SHAP harus melihat data dalam skala yang sama dengan yang dilihat model
        scaler = model.named_steps["scaler"]
        penjelas = shap.LinearExplainer(inti, scaler.transform(X_tr))
        nilai = penjelas.shap_values(scaler.transform(X_te))
    else:
        penjelas = shap.TreeExplainer(inti)
        nilai = penjelas.shap_values(X_te)
        if isinstance(nilai, list):        # sebagian versi mengembalikan per kelas
            nilai = nilai[1]
        if np.ndim(nilai) == 3:
            nilai = nilai[:, :, 1]

    return pd.Series(np.abs(nilai).mean(axis=0), index=X_te.columns)


def nilai_bawaan(model, fitur):
    """Koefisien terstandardisasi (LR) atau Gini importance (RF/XGB)."""
    inti = model.named_steps["clf"] if isinstance(model, Pipeline) else model
    if hasattr(inti, "coef_"):
        return pd.Series(inti.coef_.ravel(), index=fitur)
    if hasattr(inti, "feature_importances_"):
        return pd.Series(inti.feature_importances_, index=fitur)
    return None

In [126]:
%%time
baris = []
for varian, obj in [("utama", False), ("ekstensi", True)]:
    for t in SNAPSHOTS:
        X, y, meta = buat_snapshot(base, t, objektif=obj)
        tr, te = belah_kronologis(meta)
        X_tr, y_tr = X[tr], y[tr]
        X_te = X[te].reset_index(drop=True)
        y_te = y[te].reset_index(drop=True)

        for nama, (est, grid) in daftar_model().items():
            model, _ = cari_hyperparameter(est, grid, X_tr, y_tr)

            s_shap = nilai_shap(model, nama, X_tr, X_te)
            s_bawaan = nilai_bawaan(model, X.columns)

            perm = permutation_importance(model, X_te, y_te, scoring="roc_auc",
                                          n_repeats=20, random_state=RANDOM_STATE, n_jobs=1)
            s_perm = pd.Series(perm.importances_mean, index=X_te.columns)

            for f in X.columns:
                baris.append({
                    "varian": varian, "snapshot": t, "model": nama, "fitur": f,
                    "shap": float(s_shap[f]) if s_shap is not None else np.nan,
                    "permutasi": float(s_perm[f]),
                    "bawaan": float(s_bawaan[f]) if s_bawaan is not None else np.nan,
                })

tabel_kepentingan = pd.DataFrame(baris)
tabel_kepentingan.head()

Background dataset has 349 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=349 when initializing the masker.
Background dataset has 349 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=349 when initializing the masker.
Background dataset has 349 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=349 when initializing the masker.
Background dataset has 339 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=339 when initializing the masker.
Background dataset has 349 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=349 when initializing the masker.
Background dataset has 349 samples but max_samples=100. Subsampling to 100 samples for SHAP val

CPU times: total: 3min 43s
Wall time: 8min 38s


,varian,snapshot,model,fitur,shap,permutasi,bawaan
0,utama,10,LogisticRegression,golddiffat10,0.305251,0.059707,0.392305
1,utama,10,LogisticRegression,xpdiffat10,0.070413,-0.002318,0.092419
2,utama,10,LogisticRegression,csdiffat10,0.232508,0.022934,0.342577
3,utama,10,LogisticRegression,killdiffat10,0.160036,0.019125,0.189795
4,utama,10,LogisticRegression,assistdiffat10,0.005692,0.000271,0.006759


In [127]:
# Apakah ketiga model sepakat soal fitur dominan?
metrik = "shap" if ADA_SHAP else "permutasi"
for varian in ["utama", "ekstensi"]:
    for t in SNAPSHOTS:
        sub = tabel_kepentingan[(tabel_kepentingan["varian"] == varian) &
                                (tabel_kepentingan["snapshot"] == t)]
        piv = sub.pivot_table(index="fitur", columns="model", values=metrik)
        print(f"\n=== varian {varian} | snapshot menit {t} ({metrik}) ===")
        print(piv.abs().rank(ascending=False).astype(int)
              .sort_values(piv.columns[0]).to_string())


=== varian utama | snapshot menit 10 (shap) ===
model           LogisticRegression  RandomForest  XGBoost
fitur                                                    
golddiffat10                     1             1        1
csdiffat10                       2             2        2
killdiffat10                     3             4        5
xpdiffat10                       4             3        3
assistdiffat10                   5             5        4

=== varian utama | snapshot menit 15 (shap) ===
model           LogisticRegression  RandomForest  XGBoost
fitur                                                    
golddiffat15                     1             2        2
xpdiffat15                       2             1        1
csdiffat15                       3             3        3
killdiffat15                     4             5        5
assistdiffat15                   5             4        4

=== varian utama | snapshot menit 20 (shap) ===
model           LogisticRegression  Rando

---
## Langkah 7 — Audit temporal: first baron dan Atakhan

Pada musim 2025 Baron baru muncul pada 25:00, sedangkan dataset hanya menyimpan
apakah objektif diperoleh *kapan saja* dalam pertandingan. Karena itu
`firstbaron` pada snapshot menit 25 memuat informasi dari masa depan.

Sel di bawah menambahkan `firstbaron` dan `atakhandiff` ke snapshot menit 25
**secara terpisah**, sehingga sumber inflasi akurasi dapat ditunjuk langsung
(bukan disimpulkan dari urutan kepentingan fitur saja). Ini demonstrasi
kebocoran, bukan bagian model utama.

In [128]:
%%time
assert "firstbaron" in base.columns, "kolom firstbaron tidak ada di CSV"

hasil_audit = []
for varian, tambahan in [("utama (numerik)", []),
                         ("+ firstbaron", ["firstbaron"]),
                         ("+ atakhandiff", ["atakhandiff"]),
                         ("+ firstbaron + atakhandiff", ["firstbaron", "atakhandiff"])]:
    X, y, meta = buat_snapshot(base, 25, tambahan=tambahan)
    tr, te = belah_kronologis(meta)
    for nama, (est, grid) in daftar_model().items():
        r, _, _ = latih_dan_evaluasi(nama, est, grid,
                                     X[tr], y[tr],
                                     X[te].reset_index(drop=True),
                                     y[te].reset_index(drop=True))
        r["varian"] = varian
        hasil_audit.append(r)

tabel_audit = pd.DataFrame(hasil_audit)[["varian", "model", "n", "accuracy", "auc", "brier"]]
base_acc = ambang[25]
tabel_audit["selisih_vs_baseline_gold"] = tabel_audit["accuracy"] - base_acc
tabel_audit.round(4)

CPU times: total: 1min 14s
Wall time: 3min 39s


,varian,model,n,accuracy,auc,brier,selisih_vs_baseline_gold
0,utama (numerik),LogisticRegression,201.0,0.8159,0.9065,0.1342,-0.0050
1,utama (numerik),RandomForest,201.0,0.8164,0.9049,0.1283,-0.0045
2,utama (numerik),XGBoost,201.0,0.8219,0.9050,0.1308,0.0010
3,+ firstbaron,LogisticRegression,201.0,0.8358,0.9343,0.1164,0.0149
4,+ firstbaron,RandomForest,201.0,0.8507,0.9298,0.1115,0.0299
5,+ firstbaron,XGBoost,201.0,0.8289,0.9269,0.1197,0.0080
6,+ atakhandiff,LogisticRegression,201.0,0.8060,0.9044,0.1330,-0.0149
7,+ atakhandiff,RandomForest,201.0,0.8055,0.9061,0.1286,-0.0154
8,+ atakhandiff,XGBoost,201.0,0.8209,0.9059,0.1311,0.0000
9,+ firstbaron + atakhandiff,LogisticRegression,201.0,0.8458,0.9333,0.1104,0.0249


In [129]:
%%time
# Apakah fitur yang bocor merebut urutan teratas kepentingan fitur?
# Bila ya, itu sinyal untuk memeriksa ulang asumsi ketersediaan waktunya.
X, y, meta = buat_snapshot(base, 25, tambahan=["firstbaron", "atakhandiff"])
tr, te = belah_kronologis(meta)
X_tr, X_te = X[tr], X[te].reset_index(drop=True)
y_tr, y_te = y[tr], y[te].reset_index(drop=True)

kolom_audit = {}
for nama, (est, grid) in daftar_model().items():
    model, _ = cari_hyperparameter(est, grid, X_tr, y_tr)
    perm = permutation_importance(model, X_te, y_te, scoring="roc_auc",
                                  n_repeats=20, random_state=RANDOM_STATE, n_jobs=1)
    kolom_audit[(nama, "permutasi")] = pd.Series(perm.importances_mean, index=X_te.columns)
    s = nilai_shap(model, nama, X_tr, X_te)
    if s is not None:
        kolom_audit[(nama, "shap")] = s

kepentingan_audit = pd.DataFrame(kolom_audit)
kepentingan_audit.round(4).sort_values(kepentingan_audit.columns[0], ascending=False)

Background dataset has 338 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=338 when initializing the masker.


CPU times: total: 16.1 s
Wall time: 51.8 s


LogisticRegression         RandomForest           XGBoost        
                        permutasi    shap    permutasi    shap permutasi    shap
firstbaron                 0.0553  1.0703       0.0361  0.1120    0.0714  1.1037
csdiffat25                 0.0152  0.4392       0.0062  0.0257    0.0017  0.0445
golddiffat25               0.0114  0.4671       0.0334  0.1108    0.0753  1.0654
xpdiffat25                 0.0087  0.5232       0.0282  0.1221    0.0285  0.9058
killdiffat25               0.0062  0.3325       0.0082  0.0432   -0.0001  0.0019
assistdiffat25             0.0047  0.2875       0.0015  0.0455    0.0016  0.0863
atakhandiff               -0.0005  0.3727      -0.0026  0.0241   -0.0001  0.0633

In [130]:
import os
os.makedirs("outputs", exist_ok=True)

semua_tabel = {
    "tabel2_baseline": tabel2,
    "hasil_utama": tabel_utama,
    "hiperparameter_terpilih": hasil_utama[["snapshot", "model", "params"]],
    "perbandingan_baseline": banding,
    "delong": tabel_delong,
    "keandalan": tabel_keandalan,
    "varian_ekstensi_biner": hasil_ekstensi,
    "perbandingan_utama_vs_ekstensi": tabel4,
    "sensitivitas_subset": hasil_sensitivitas,
    "subkelompok_playoff": tabel_subkelompok,
    "ringkasan_subkelompok": ringkas_sub,
    "uji_playoff": tabel_uji_playoff,
    "kebocoran": hasil_kebocoran,
    "audit_baron_atakhan": tabel_audit,
    "kepentingan_audit": kepentingan_audit,
    "kepentingan_fitur": tabel_kepentingan,
}

for nama, df in semua_tabel.items():
    df.to_csv(f"outputs/{nama}.csv", index=True)
    print(f"outputs/{nama}.csv  ({len(df)} baris)")

outputs/tabel2_baseline.csv  (4 baris)
outputs/hasil_utama.csv  (12 baris)
outputs/hiperparameter_terpilih.csv  (12 baris)
outputs/perbandingan_baseline.csv  (12 baris)
outputs/delong.csv  (12 baris)
outputs/keandalan.csv  (3 baris)
outputs/varian_ekstensi_biner.csv  (12 baris)
outputs/perbandingan_utama_vs_ekstensi.csv  (12 baris)
outputs/sensitivitas_subset.csv  (12 baris)
outputs/subkelompok_playoff.csv  (36 baris)
outputs/ringkasan_subkelompok.csv  (12 baris)
outputs/uji_playoff.csv  (12 baris)
outputs/kebocoran.csv  (6 baris)
outputs/audit_baron_atakhan.csv  (12 baris)
outputs/kepentingan_audit.csv  (7 baris)
outputs/kepentingan_fitur.csv  (162 baris)


---
## Langkah 8 — Gambar untuk naskah

Gambar dibangun langsung dari dataframe hasil, bukan diketik ulang, sehingga
ikut berubah bila eksperimen dijalankan ulang. Garis dibedakan oleh jenis garis
dan bentuk penanda, bukan hanya warna, agar terbaca saat dicetak hitam putih.
Berkas disimpan dalam PNG, SVG, dan PDF, dalam dua bahasa.

In [131]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import os

os.makedirs("gambar", exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.linewidth": 0.8})

BAHASA = ("id", "en")          # ubah jadi ("en",) kalau hanya perlu Inggris

GAYA = {"LogisticRegression": ("-", "o"),
        "RandomForest": ("--", "s"),
        "XGBoost": ("-.", "^")}
LABEL = {"LogisticRegression": "Logistic Regression",
         "RandomForest": "Random Forest",
         "XGBoost": "XGBoost"}

# Seluruh teks yang berbeda antarbahasa dikumpulkan di satu tempat,
# sehingga menambah bahasa lain cukup dengan menambah satu entri.
TEKS = {
"id": {
  "dec": ",",
  "menit": "Menit pertandingan",
  "snapshot_x": "Snapshot (menit)",
  "akurasi": "Akurasi (%)", "auc": "AUC-ROC", "brier": "Brier score",
  "p_a": "(a) Akurasi terhadap baseline",
  "p_b": "(b) AUC-ROC",
  "p_c": "(c) Brier score (makin rendah makin baik)",
  "baseline": "Baseline tanda gold",
  "baseline_lr": "Regresi logistik selisih gold",
  "ambang": "ambang AUC {:.2f}",
  "sens": "hanya analisis sensitivitas", "tidak": "tidak dipakai",
  "penurunan": "Penurunan AUC saat fitur diacak",
  "leg_num": "Fitur numerik selisih", "leg_bin": "Indikator objektif (varian sensitivitas)",
  "wr": "Kemenangan blue side (%)", "latih": "(latih)", "uji": "(uji)",
  "akhir": "akhir",
  "fitur": {"golddiffat": "Selisih gold", "xpdiffat": "Selisih experience",
            "csdiffat": "Selisih CS", "killdiffat": "Selisih kill",
            "assistdiffat": "Selisih assist", "grubdiff": "Selisih void grubs",
            "firstblood": "First blood", "firstdragon": "First dragon",
            "firstherald": "First herald"},
  "objektif": {"First dragon": "First dragon", "Void grub difference": "Selisih void grubs",
               "First herald": "First herald", "First tower": "First tower",
               "Atakhan": "Atakhan", "First baron": "First baron"},
},
"en": {
  "dec": ".",
  "menit": "Match minute",
  "snapshot_x": "Snapshot (minute)",
  "akurasi": "Accuracy (%)", "auc": "AUC-ROC", "brier": "Brier score",
  "p_a": "(a) Accuracy against the baseline",
  "p_b": "(b) AUC-ROC",
  "p_c": "(c) Brier score (lower is better)",
  "baseline": "Gold difference sign baseline",
  "baseline_lr": "Gold difference logistic regression",
  "ambang": "AUC threshold {:.2f}",
  "sens": "sensitivity analysis only", "tidak": "not used",
  "penurunan": "Drop in AUC when the feature is permuted",
  "leg_num": "Numeric difference features", "leg_bin": "Objective indicators (sensitivity variant)",
  "wr": "Blue side win rate (%)", "latih": "(train)", "uji": "(test)",
  "akhir": "end",
  "fitur": {"golddiffat": "Gold difference", "xpdiffat": "Experience difference",
            "csdiffat": "CS difference", "killdiffat": "Kill difference",
            "assistdiffat": "Assist difference", "grubdiff": "Void grub difference",
            "firstblood": "First blood", "firstdragon": "First dragon",
            "firstherald": "First herald"},
  "objektif": {"First dragon": "First dragon", "Void grub difference": "Void grub difference",
               "First herald": "First herald", "First tower": "First tower",
               "Atakhan": "Atakhan", "First baron": "First baron"},
},
}


def angka(v, lang, fmt="{:.4f}"):
    if abs(v) < 5e-5:
        v = 0.0
    """Desimal koma untuk Indonesia, titik untuk Inggris."""
    return fmt.format(v).replace(".", TEKS[lang]["dec"])


def simpan(fig, nama, lang):
    for ext in ("png", "svg", "pdf"):
        kw = {"dpi": 300} if ext == "png" else {}
        fig.savefig(f"gambar/{nama}_{lang}.{ext}", bbox_inches="tight",
                    pad_inches=0.05, facecolor="white", **kw)
    print(f"gambar/{nama}_{lang}.[png|svg|pdf]")
    plt.close(fig)

In [132]:
AKHIR = 30

# PERIKSA jendela waktu di bawah terhadap Tabel I naskah DAN patch notes 15.x
# sebelum gambar dipakai. Ketidakcocokan yang diketahui:
#   - First herald: Tabel I naskah 16:00-24:45, di sini 15:00-24:45
#   - Void grub: Tabel I naskah "spawn pertama 6:00, kedua 15:00", di sini 6:00-13:45
JENDELA = [   # (kunci teks, mulai, selesai, status, keterangan)
    ("First dragon",         5.0,  AKHIR, "sens",  "5:00 - {akhir}"),
    ("Void grub difference", 6.0,  AKHIR, "sens", "6:00 (8:00 from 15.09) - {akhir}"),
    ("First herald",         15.0, AKHIR, "sens", "15:00 (16:00 before 15.09) - {akhir}"),
    ("First tower",          14.0, AKHIR, "tidak", "14:00 - {akhir}"),
    ("Atakhan",              20.0, AKHIR, "tidak", "20:00 - {akhir}"),
    ("First baron",          25.0, AKHIR, "tidak", "25:00 - {akhir}"),
]


def gambar_jendela(lang):
    T = TEKS[lang]
    fig, ax = plt.subplots(figsize=(7.8, 3.6))
    for i, (kunci, mulai, selesai, status, ket) in enumerate(JENDELA):
        y = len(JENDELA) - i
        sens = status == "sens"
        # abu-abu polos = analisis sensitivitas, arsir = tidak dipakai;
        # aman dicetak hitam-putih
        ax.barh(y, selesai - mulai, left=mulai, height=0.55,
                color="0.82" if sens else "0.94", edgecolor="black",
                linewidth=0.9, hatch="" if sens else "///", zorder=2)
        ax.text(mulai + 0.3, y, ket.format(akhir=T["akhir"]), va="center",
                fontsize=8, zorder=4,
                bbox=dict(facecolor="white", edgecolor="none", pad=1.0, alpha=0.92))
        ax.text(AKHIR + 0.8, y, T[status], va="center", fontsize=8.5,
                fontweight="normal" if sens else "bold")
    for t in SNAPSHOTS:
        ax.axvline(t, color="0.35", ls="--", lw=1.0, zorder=1)
        ax.text(t, len(JENDELA) + 0.75, f"t={t}", ha="center",
                fontsize=9.5, fontweight="bold")
    ax.set_yticks(range(1, len(JENDELA) + 1))
    ax.set_yticklabels([T["objektif"][k] for k, *_ in JENDELA][::-1], fontsize=9.5)
    ax.set_xlim(0, AKHIR + 11); ax.set_ylim(0.3, len(JENDELA) + 1.3)
    ax.set_xticks(range(0, AKHIR + 1, 5))
    ax.set_xlabel(T["menit"])
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    simpan(fig, "fig3_objective_windows", lang)


for lang in BAHASA:
    gambar_jendela(lang)

gambar/fig3_objective_windows_id.[png|svg|pdf]
gambar/fig3_objective_windows_en.[png|svg|pdf]


In [133]:
def gambar_performa(lang):
    T = TEKS[lang]
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.4))
    panel = [("accuracy", T["akurasi"], T["p_a"], 100),
             ("auc",      T["auc"],     T["p_b"], 1),
             ("brier",    T["brier"],   T["p_c"], 1)]
    for k, (kolom, ylabel, judul, skala) in enumerate(panel):
        for m, (ls, mk) in GAYA.items():
            sub = tabel_utama[tabel_utama["model"] == m].sort_values("snapshot")
            ax[k].plot(sub["snapshot"], sub[kolom] * skala, ls, marker=mk,
                       color="black", ms=5, lw=1.2, mfc="white", label=LABEL[m])
        ax[k].set_ylabel(ylabel)
        ax[k].set_title(judul, fontsize=10)

    bl = baseline_uji.sort_values("snapshot")
    ax[0].plot(bl["snapshot"], bl["tanda_gold"] * 100, ":", marker="D",
               color="0.35", ms=5, lw=1.6, label=T["baseline"])
    ax[1].axhline(AUC_MIN, color="0.35", ls=":", lw=1.2)
    ax[1].text(SNAPSHOTS[0] + 0.3, AUC_MIN + 0.006,
               T["ambang"].format(AUC_MIN).replace(".", T["dec"]),
               fontsize=8, color="0.25")

    bp = tabel_bp.sort_values("snapshot")
    ax[1].plot(bp["snapshot"], bp["auc"], "-", marker="s", color="0.55",
               ms=4, lw=1.4, label=T["baseline_lr"])
    ax[2].plot(bp["snapshot"], bp["brier"], "-", marker="s", color="0.55",
               ms=4, lw=1.4, label=T["baseline_lr"])
    ax[0].plot(bp["snapshot"], bp["accuracy"] * 100, "-", marker="s", color="0.55",
               ms=4, lw=1.4, label=T["baseline_lr"])
    
    for a in ax:
        a.set_xticks(SNAPSHOTS); a.set_xlabel(T["snapshot_x"])
        a.grid(True, ls=":", lw=0.5, alpha=0.5)
        a.spines["top"].set_visible(False); a.spines["right"].set_visible(False)
    h, l = ax[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=5, frameon=False,
               fontsize=9, bbox_to_anchor=(0.5, 0.0),
               columnspacing=1.6, handlelength=2.4)
    fig.tight_layout(rect=(0, 0.10, 1, 1))   # sisakan 10% tinggi di bawah untuk legend
    simpan(fig, "fig4_model_performance", lang)


for lang in BAHASA:
    gambar_performa(lang)

gambar/fig4_model_performance_id.[png|svg|pdf]
gambar/fig4_model_performance_en.[png|svg|pdf]


In [134]:
SNAPSHOT_GAMBAR = 20


def gambar_kepentingan(lang, snapshot=SNAPSHOT_GAMBAR):
    """Permutation importance dari varian EKSTENSI, agar kontras numerik vs biner terlihat."""
    T = TEKS[lang]
    sub = tabel_kepentingan[(tabel_kepentingan["snapshot"] == snapshot) &
                            (tabel_kepentingan["varian"] == "ekstensi")]
    rata = sub.pivot_table(index="fitur", columns="model",
                           values="permutasi").mean(axis=1).sort_values()
    biner = set(FITUR_OBJEKTIF[snapshot])
    label = [T["fitur"].get(f.replace(str(snapshot), ""), f) for f in rata.index]
    warna = ["0.75" if f in biner else "0.2" for f in rata.index]

    fig, ax = plt.subplots(figsize=(6.2, 3.2))
    ax.barh(range(len(rata)), rata.values, color=warna, height=0.62,
            edgecolor="black", linewidth=0.5)
    ax.set_yticks(range(len(rata)))
    ax.set_yticklabels(label, fontsize=9.5)
    ax.set_xlabel(T["penurunan"])
    ax.grid(True, axis="x", ls=":", lw=0.5, alpha=0.6)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    batas = max(rata.values) * 1.22
    for i, v in enumerate(rata.values):
        ax.text(v + batas * 0.02, i, angka(v, lang), va="center", fontsize=8.5)
    ax.set_xlim(min(0, min(rata.values) * 1.2), batas)
    ax.legend(handles=[Patch(facecolor="0.2",  edgecolor="black", label=T["leg_num"]),
                       Patch(facecolor="0.75", edgecolor="black", label=T["leg_bin"])],
              loc="lower right", fontsize=8.5, frameon=False)
    plt.tight_layout()
    simpan(fig, f"fig5_permutation_importance_t{snapshot}", lang)


for lang in BAHASA:
    gambar_kepentingan(lang)

gambar/fig5_permutation_importance_t20_id.[png|svg|pdf]
gambar/fig5_permutation_importance_t20_en.[png|svg|pdf]


In [135]:
def gambar_baserate(lang):
    T = TEKS[lang]
    urut = ["Cup", "Rounds 1-2", "Rounds 3-5"]
    wr = base.groupby("split")[TARGET].agg(["mean", "size"]).reindex(urut)

    fig, ax = plt.subplots(figsize=(5.2, 3.0))
    ax.plot(range(len(urut)), wr["mean"] * 100, "-o", color="black",
            ms=7, lw=1.4, mfc="white")
    ax.axhline(50, color="0.55", ls=":", lw=1)
    for i, (m, n) in enumerate(zip(wr["mean"], wr["size"])):
        ax.text(i, m * 100 + 1.2, angka(m * 100, lang, "{:.1f}") + "%",
                ha="center", fontsize=9)
        ax.text(i, m * 100 - 2.4, f"n={n}", ha="center", fontsize=8, color="0.4")
    ax.set_xticks(range(len(urut)))
    ax.set_xticklabels([f"LCK Cup\n{T['latih']}", f"Rounds 1-2\n{T['latih']}",
                        f"Rounds 3-5\n{T['uji']}"], fontsize=9)
    ax.set_ylabel(T["wr"]); ax.set_ylim(40, 68)
    ax.grid(True, axis="y", ls=":", lw=0.5, alpha=0.6)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    simpan(fig, "fig6_base_rate_shift", lang)


for lang in BAHASA:
    gambar_baserate(lang)

gambar/fig6_base_rate_shift_id.[png|svg|pdf]
gambar/fig6_base_rate_shift_en.[png|svg|pdf]


In [137]:
g = (base["void_grubs"] + base["opp_void_grubs"]) / 2
print(g.groupby(base["patch"].astype(float) >= 15.09).mean())
print(base.loc[base["barons"] + base["opp_barons"] > 0, "gamelength"].min() / 60)

patch
False    2.997826
True     1.495385
dtype: float64
26.283333333333335


In [138]:
d = base["gamelength"] / 60
band = base[(d >= 26) & (d < 27)]
print(len(band), ((band["barons"] + band["opp_barons"]) > 0).mean())

29 0.41379310344827586
